# 03 · Model — who to call

**Question:** if counsellors call leads in score order, how many enrolments do they reach per call?

Rules: every *choice* (class weights, LightGBM settings, A vs B, calibration method, band cutoffs) is made with
5-fold cross-validation on the training leads. The test leads are used once, for the numbers we report.
Logic: `src/model.py`, `src/metrics.py`.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             brier_score_loss, log_loss)

from src.config import FIG_DIR, ROOT as REPO, load_config
from src.clean import load_raw, LeadCleaner, TARGET
from src import model as M, metrics, eda

pd.set_option("display.max_colwidth", 120)
cfg = load_config()
RS = cfg["model"]["random_state"]
MODELS = REPO / "models"; MODELS.mkdir(exist_ok=True)

df = load_raw()
train, test = M.split(df, cfg)
cleaner = LeadCleaner().fit(train)
X_train, X_test = cleaner.transform(train), cleaner.transform(test)
y_train, y_test = train[TARGET].to_numpy(), test[TARGET].to_numpy()
BASE = y_test.mean()
print(f"train {X_train.shape}, test {X_test.shape}, test baseline conversion {BASE:.1%}")

train (6930, 54), test (2310, 54), test baseline conversion 38.5%


## 1. Model A — logistic regression

In [2]:
cw, cw_table = M.select_logistic(X_train, y_train, cfg)
display(cw_table)
CW = None if cw == "None" else cw
print(f"class_weight = {cw} (higher out-of-fold PR-AUC)")
model_a = M.logistic(CW, RS)

,PR-AUC (OOF),precision@20% (OOF)
None,0.821491,0.871573
balanced,0.821446,0.871573


class_weight = None (higher out-of-fold PR-AUC)


### Odds ratios (fitted on all training leads)
For yes/no features: how much higher the odds of enrolling are when the feature is *yes*.
For numeric features: per one standard deviation.

In [3]:
fit_a = clone(model_a).fit(X_train, y_train)
orr = M.odds_ratios(fit_a, X_train)
orr.to_csv(REPO / "reports" / "odds_ratios.csv", index=False)
display(orr.head(15)[["label", "unit", "odds_ratio", "sd"]])

,label,unit,odds_ratio,sd
17,Lead source: Welingak Website,yes vs no,14.009344,0.126097
9,Lead origin: Lead Add Form,yes vs no,9.892584,0.269858
42,Occupation: Working Professional,yes vs no,7.049340,0.266222
10,Lead origin: other / rare,yes vs no,0.197068,0.081202
5,Opted out of email,yes vs no,0.253316,0.273426
2,Time on website,per 1 SD,3.175208,543.431608
44,Main motivation: Better Career Prospects,yes vs no,3.165803,0.453313
45,Main motivation left blank,yes vs no,0.316147,0.453179
46,Main motivation: other / rare,yes vs no,0.333287,0.016986
40,Occupation: Student,yes vs no,0.340317,0.151097


## 2. Model B — LightGBM, light tuning (≤20 trials, 5-fold CV)

In [4]:
best_params, study = M.tune_lightgbm(X_train, y_train, cfg)
print(f"{len(study.trials)} trials, best out-of-fold PR-AUC {study.best_value:.3f}")
print(best_params)
model_b = M.lightgbm(RS, **best_params)

C:\Users\Ruthik\Desktop\counsellor-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20 trials, best out-of-fold PR-AUC 0.836
{'n_estimators': 150, 'learning_rate': 0.032149916325674675, 'num_leaves': 32, 'min_child_samples': 38, 'subsample': 0.9005190383280673, 'colsample_bytree': 0.5022170738340744, 'reg_lambda': 0.04214279993325537}


## 3. A vs B on lift in the top 2 deciles (out-of-fold, training leads)

In [5]:
winner, ab = M.compare_a_b(X_train, y_train, model_a, model_b, cfg)
display(ab)
gain = ab.attrs["gain_pp"]
margin = cfg["model"]["min_gain_pp_top20"]
CHOSEN = {"A": "Logistic regression", "B": "LightGBM"}[winner]
chosen_est = model_a if winner == "A" else model_b
why = (f"LightGBM's top-20% precision is {gain:+.1f} pp vs logistic regression; the bar set in config "
       f"before comparing was +{margin:.1f} pp. ")
why += ("Not enough to justify a less explainable model → keep logistic regression."
        if winner == "A" else "That clears the bar → use LightGBM.")
print("Chosen:", CHOSEN); print(why)

,precision@20% (OOF),lift@20% (OOF),PR-AUC (OOF)
A: logistic,0.871573,2.261325,0.821491
B: LightGBM,0.889610,2.308124,0.835960


Chosen: Logistic regression
LightGBM's top-20% precision is +1.8 pp vs logistic regression; the bar set in config before comparing was +2.0 pp. Not enough to justify a less explainable model → keep logistic regression.


## 4. Calibrate (the economics in Phase 4 multiply these probabilities by money)

In [6]:
cal_method, cal_table, oof_cal = M.choose_calibration(chosen_est, X_train, y_train, cfg)
display(cal_table)
print(f"Calibration: {cal_method} (lower out-of-fold Brier score)")

calibrated = CalibratedClassifierCV(clone(chosen_est), method=cal_method,
                                    cv=cfg["model"]["cv_folds"]).fit(X_train, y_train)
base_model = clone(chosen_est).fit(X_train, y_train)      # used for SHAP explanations

p_test = calibrated.predict_proba(X_test)[:, 1]
p_test_raw = base_model.predict_proba(X_test)[:, 1]
rel = {f"calibrated ({cal_method})": M.reliability_table(y_test, p_test),
       "uncalibrated": M.reliability_table(y_test, p_test_raw)}
b_cal, b_raw = brier_score_loss(y_test, p_test), brier_score_loss(y_test, p_test_raw)
worst = (rel[f"calibrated ({cal_method})"].mean_predicted - rel[f"calibrated ({cal_method})"].observed).abs().max()
title = (f"Calibrated scores track reality: across test-score deciles predicted and actual conversion "
         f"differ by at most {worst*100:.0f} pp (Brier {b_cal:.3f} vs {b_raw:.3f} uncalibrated)")
eda.plot_reliability(rel, title, FIG_DIR / "03_reliability.png")
display(rel[f"calibrated ({cal_method})"])

,Brier (OOF),mean predicted,actual rate
sigmoid,0.138525,0.385697,0.385426
isotonic,0.138000,0.385491,0.385426


Calibration: isotonic (lower out-of-fold Brier score)


,mean_predicted,observed,leads
bin,,,
0,0.021344,0.051948,231
1,0.068107,0.060606,231
2,0.100897,0.086580,231
3,0.149407,0.194805,231
4,0.218748,0.229437,231
5,0.332199,0.380952,231
6,0.517389,0.528139,231
7,0.697358,0.679654,231
8,0.781559,0.735931,231


## 5. Lead score 0–100 and Hot / Warm / Cold bands

In [7]:
bands_cfg = cfg["model"]["bands"]
cutoffs = M.band_cutoffs(M.to_score(oof_cal), bands_cfg["hot"], bands_cfg["warm"])
print(f"Cutoffs from out-of-fold training scores: Hot ≥ {cutoffs['hot']:.1f}, "
      f"Warm ≥ {cutoffs['warm']:.1f}, Cold < {cutoffs['warm']:.1f}")

s_test = M.to_score(p_test)
band_test = M.assign_band(s_test, cutoffs)
bt = (pd.DataFrame({"band": band_test, "y": y_test, "score": s_test})
      .groupby("band").agg(leads=("y", "size"), conversions=("y", "sum"),
                           conv_rate=("y", "mean"), avg_score=("score", "mean"))
      .reindex(["Hot", "Warm", "Cold"]))
bt["share_of_leads"] = bt.leads / bt.leads.sum()
bt["share_of_conversions"] = bt.conversions / bt.conversions.sum()
display(bt)
bt.to_csv(REPO / "reports" / "bands_test.csv")

Cutoffs from out-of-fold training scores: Hot ≥ 74.0, Warm ≥ 29.9, Cold < 29.9


,leads,conversions,conv_rate,avg_score,share_of_leads,share_of_conversions
band,,,,,,
Hot,478,389,0.813808,86.797699,0.206926,0.437079
Warm,672,355,0.528274,51.187202,0.290909,0.398876
Cold,1160,146,0.125862,11.250345,0.502165,0.164045


## 6. Business metrics on the test leads

In [8]:
atk = metrics.at_k_table(y_test, p_test, ks=(0.1, 0.2, 0.3, 0.4))
display(atk)
atk.to_csv(REPO / "reports" / "precision_recall_at_k.csv")
dec = metrics.lift_by_decile(y_test, p_test)
display(dec)
dec.to_csv(REPO / "reports" / "lift_by_decile.csv")

,leads_called,precision,recall,lift,calls_per_enrolment,random_calls_per_enrolment
top_k,,,,,,
10%,231,0.904762,0.234831,2.348315,1.105263,2.595506
20%,462,0.820346,0.425843,2.129213,1.218997,2.595506
30%,693,0.774892,0.603371,2.011236,1.290503,2.595506
40%,924,0.712121,0.739326,1.848315,1.404255,2.595506


,leads,conversions,rate,lift,cum_recall
decile,,,,,
1,231,209.0,0.904762,2.348315,0.234831
2,231,170.0,0.735931,1.910112,0.425843
3,231,158.0,0.683983,1.775281,0.603371
4,231,121.0,0.523810,1.359551,0.739326
5,231,88.0,0.380952,0.988764,0.838202
6,231,53.0,0.229437,0.595506,0.897753
7,231,44.0,0.190476,0.494382,0.947191
8,231,16.0,0.069264,0.179775,0.965169
9,231,18.0,0.077922,0.202247,0.985393


In [9]:
K = 0.2
row = atk.loc[f"{K:.0%}"]
HEADLINE = (f"Top {K:.0%} of leads capture {row.recall:.0%} of conversions at {row.lift:.1f}× baseline")
print(HEADLINE)
print(f"Calls per enrolment in the top {K:.0%}: {row.calls_per_enrolment:.2f} (model) vs "
      f"{row.random_calls_per_enrolment:.2f} (random order)")
g = metrics.gains_curve(y_test, p_test)
eda.plot_gains(g, HEADLINE + f" ({len(y_test):,} held-out test leads)", mark_k=K,
               path=FIG_DIR / "03_gains.png")

Top 20% of leads capture 43% of conversions at 2.1× baseline
Calls per enrolment in the top 20%: 1.22 (model) vs 2.60 (random order)


<Figure size 700x500 with 1 Axes>

## 7. What drives the score? (SHAP)

In [10]:
bg = X_train.sample(500, random_state=RS)
sv_test = M.shap_values(base_model, bg, X_test)
glob = M.global_drivers(sv_test, X_test, top=10)
display(glob)
glob.to_csv(REPO / "reports" / "global_drivers.csv", index=False)
t0 = glob.iloc[0]
title = (f"Biggest score driver: {t0.label} ({'raises' if t0.direction == 'raises' else 'lowers'} the odds); "
         f"top 10 drivers on test leads")
eda.plot_drivers(glob, title, FIG_DIR / "03_shap_top10.png")

,feature,label,importance,direction
0,Total Time Spent on Website,Time on website,0.996253,raises
1,What matters most to you in choosing a course=Better Career Prospects,Main motivation: Better Career Prospects,0.491654,raises
2,What matters most to you in choosing a course=Missing,Main motivation left blank,0.491097,lowers
3,Lead Origin=Landing Page Submission,Lead origin: Landing Page Submission,0.393863,lowers
4,Lead Origin=Lead Add Form,Lead origin: Lead Add Form,0.356288,raises
5,What is your current occupation=Unemployed,Occupation: Unemployed,0.340320,lowers
6,Country=Missing,Country left blank,0.301848,raises
7,Country=India,Country: India,0.253598,lowers
8,What is your current occupation=Working Professional,Occupation: Working Professional,0.236675,raises
9,Specialization=Missing,Specialization left blank,0.236606,lowers


<Figure size 800x560 with 1 Axes>

### Per-lead top-3 drivers (stored for the call briefs)

In [11]:
scorer = M.LeadScorer(cleaner=cleaner, calibrated=calibrated, base_model=base_model,
                      model_name=CHOSEN, calibration=cal_method, cutoffs=cutoffs, background=bg,
                      meta={"class_weight": cw, "lgbm_params": best_params,
                            "feature_names": cleaner.feature_names_})
joblib.dump(scorer, MODELS / "scorer.joblib")

scored = []
for name, part in (("train", train), ("test", test)):
    s = scorer.score(part)
    s.insert(0, "split", name)
    scored.append(s)
scored = pd.concat(scored).loc[df.index]
out = df[["Prospect ID", "Lead Number", "Do Not Call", TARGET]].join(scored)
out["drivers"] = out["drivers"].apply(json.dumps)
out.to_csv(MODELS / "scored_leads.csv", index=False)

hot_test = out[(out.split == "test") & (out.band == "Hot")].sort_values("score", ascending=False)
display(hot_test[["Prospect ID", "score", "band", "drivers_text", TARGET]].head(8))

,Prospect ID,score,band,drivers_text,Converted
120,144807db-2895-4002-b52e-3eda79c22395,99.2,Hot,Lead origin: Lead Add Form ↑; Occupation: Working Professional ↑; Time on website: 19 min ↑,1
7927,fbed4f56-3231-487b-a5cc-ffdb97cfa897,99.2,Hot,Lead origin: Lead Add Form ↑; Occupation: Working Professional ↑; Time on website: 0 min ↓,1
8074,20bae5db-2d10-4d69-90d0-65a389adb418,99.2,Hot,Time on website: 28 min ↑; Lead origin: Lead Add Form ↑; Occupation: Working Professional ↑,1
8099,27eff01f-2177-452f-81c9-94b132634e42,99.2,Hot,Lead origin: Lead Add Form ↑; Occupation: Working Professional ↑; Time on website: 0 min ↓,1
8118,80f5897b-9092-40f4-86e3-029d988ba27b,99.2,Hot,Lead origin: Lead Add Form ↑; Occupation: Working Professional ↑; Time on website: 0 min ↓,1
7327,95d1590f-7c47-4f40-9806-388f4472d3a4,99.2,Hot,Lead source: Welingak Website ↑; Lead origin: Lead Add Form ↑; Time on website: 16 min ↑,1
6913,cbad4d40-4228-486e-868c-f2bb6c709731,99.2,Hot,Time on website: 26 min ↑; Occupation: Working Professional ↑; Main motivation: Better Career Prospects ↑,1
1758,91e4fcd1-547f-45fd-9cd9-f679c1c27440,99.2,Hot,Lead source: Welingak Website ↑; Lead origin: Lead Add Form ↑; Time on website: 0 min ↓,1


## 8. Save the model card

In [12]:
card = {
    "model": CHOSEN, "calibration": cal_method, "class_weight": cw,
    "decision": why, "ab_table": ab.round(4).to_dict(),
    "band_cutoffs": cutoffs, "headline": HEADLINE,
    "headline_numbers": {"k": K, "recall": float(row.recall), "lift": float(row.lift),
                         "precision": float(row.precision),
                         "calls_per_enrolment_model": float(row.calls_per_enrolment),
                         "calls_per_enrolment_random": float(row.random_calls_per_enrolment)},
    "test_leads": int(len(y_test)), "test_baseline": float(BASE),
}
(MODELS / "model_card.json").write_text(json.dumps(card, indent=2), encoding="utf-8")
print(json.dumps(card["headline_numbers"], indent=2))

{
  "k": 0.2,
  "recall": 0.4258426966292135,
  "lift": 2.1292134831460676,
  "precision": 0.8203463203463204,
  "calls_per_enrolment_model": 1.2189973614775724,
  "calls_per_enrolment_random": 2.595505617977528
}


## Appendix — technical metrics (test leads)

In [13]:
fit_b = clone(model_b).fit(X_train, y_train)
p_b = fit_b.predict_proba(X_test)[:, 1]
def tech(y, p):
    return {"ROC AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "Accuracy @0.5": accuracy_score(y, p >= 0.5), "Brier": brier_score_loss(y, p),
            "Log loss": log_loss(y, p), "Precision @20%": metrics.precision_at_k(y, p, 0.2)}
app = pd.DataFrame({f"{CHOSEN} (calibrated, used)": tech(y_test, p_test),
                    "Logistic (uncalibrated)": tech(y_test, fit_a.predict_proba(X_test)[:, 1]),
                    "LightGBM (uncalibrated, not used)": tech(y_test, p_b)}).T
display(app.round(3))
app.to_csv(REPO / "reports" / "appendix_technical_metrics.csv")

,ROC AUC,PR-AUC,Accuracy @0.5,Brier,Log loss,Precision @20%
"Logistic regression (calibrated, used)",0.851,0.782,0.790,0.150,0.470,0.820
Logistic (uncalibrated),0.852,0.784,0.789,0.152,0.472,0.823
"LightGBM (uncalibrated, not used)",0.864,0.804,0.794,0.143,0.444,0.851
